# Business pilot: recorded inventory and open orders

Run **Runtime → Run all** in Google Colab. This CPU notebook demonstrates the business-data contract using an explicitly synthetic fixture. It validates physical inventory conservation, compiles only facts known at each cutoff, and compares allocation methods on identical snapshots. It does not place orders, reserve inventory, or dispatch goods.

An open-order update is an **absolute remaining quantity**, not a new demand increment. Physical receipts and dispatches are a separate ledger. Inventory snapshots are observations, never additional supply lots. The fixture is an executable intake example; its penalty scores are not measured business savings.


In [ ]:
import importlib
import importlib.util
import subprocess
import sys

required = ("load_business_pilot", "compile_business_pilot", "compare_business_pilot")
needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops
    needs_install = not all(hasattr(assumption_ops, name) for name in required)
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    for name in list(sys.modules):
        if name == "assumption_ops" or name.startswith("assumption_ops."):
            del sys.modules[name]
    importlib.invalidate_caches()

import json
import tempfile
from pathlib import Path
from urllib.request import urlopen
from IPython.display import Markdown, display
import assumption_ops
from assumption_ops import load_business_pilot, compile_business_pilot, compare_business_pilot
from assumption_ops.business_io import write_business_csv


## 1. Load an explicit business-data bundle

The local checkout is used when available. Colab downloads the six named fixture files from this repository. To evaluate your own exports, replace `manifest_path` with your bundle's manifest; the intake contract and required column meanings are documented in `docs/business_pilot.md`. Do not mark real data as synthetic or infer missing requested demand from fulfilled sales.


In [ ]:
fixture_files = (
    "manifest.json", "catalog.csv", "stock_snapshots.csv", "movements.csv",
    "open_orders.csv", "policy.json",
)
fixture_dir = Path(assumption_ops.__file__).resolve().parents[2] / "examples" / "business_pilot"
if not (fixture_dir / "manifest.json").is_file():
    fixture_dir = Path(tempfile.mkdtemp(prefix="assumption-business-pilot-"))
    base_url = "https://raw.githubusercontent.com/crypticsymmetry/improved-giggle/main/examples/business_pilot/"
    for filename in fixture_files:
        with urlopen(base_url + filename, timeout=30) as response:
            data = response.read(1_000_001)
        if len(data) > 1_000_000:
            raise ValueError(f"Unexpectedly large fixture file: {filename}")
        (fixture_dir / filename).write_bytes(data)
manifest_path = fixture_dir / "manifest.json"
pilot = load_business_pilot(manifest_path)
print("Loaded:", manifest_path)
print("Fixture is synthetic; no external execution occurs.")


## 2. Compile admitted facts at each observation cutoff

The opening stock is A=4, B=4. Day two has a receipt of two A and a dispatch of one B; day three dispatches one A. The resulting stocks must be A=6/B=3 and A=5/B=3. Those stocks are current physical capacity, rather than the sum of repeated observations.

Order `new-A` happened on day two but was observed on day three. It must remain hidden from the day-two plan. Cancellation sets the constrained order's remaining quantity to zero independently of dispatch records. Current stock is available on the cutoff date, so past-due orders incur lateness rather than being backdated to opening stock.


In [ ]:
snapshots = compile_business_pilot(pilot)
for snapshot in snapshots:
    print(snapshot.snapshot_id, snapshot.cutoff)
    print("Inventory:", {s.sku: s.quantity for s in snapshot.supplies})
    print("Open backlog:", {o.id: o.quantity for o in snapshot.orders})
    print("Admitted metadata:", snapshot.metadata)


In [ ]:
assert len(snapshots) == 3
expected_stock = ({"A": 4, "B": 4}, {"A": 6, "B": 3}, {"A": 5, "B": 3})
for snapshot, expected in zip(snapshots, expected_stock, strict=True):
    assert {s.sku: s.quantity for s in snapshot.supplies} == expected
assert "new-A" not in {o.id for o in snapshots[0].orders}
assert "new-A" not in {o.id for o in snapshots[1].orders}
assert {o.id: o.quantity for o in snapshots[2].orders}.get("new-A") == 2
assert {o.id: o.quantity for o in snapshots[1].orders} == {"constrained-A": 3, "flexible-B": 5}
assert {o.id: o.quantity for o in snapshots[2].orders}.get("constrained-A", 0) == 0
print("Physical conservation, absolute order updates, and observation-time visibility verified.")


## 3. Compare methods on identical admitted snapshots

MILP, verified LP, earliest-due-date, and priority/cost greedy receive the same capacity, open backlog, substitutions, and weights. An independently verified LP bound helps assess optimality. Timing here is illustrative for a tiny fixture.

The opening example includes a shared-SKU opportunity cost: A costs less than B but is also needed by the constrained A order. Greedy methods can spend cheap A on the flexible B order too soon. This constructed example demonstrates the mechanism, not its prevalence in a real business.


In [ ]:
report = compare_business_pilot(pilot)
assert report["synthetic"] is True
rows = []
for snapshot_record in report["snapshots"]:
    comparison = snapshot_record["comparison"]
    assert comparison["external_dispatch"] is False
    for result in comparison["results"]:
        metrics = result["evaluation"]["metrics"]
        rows.append({
            "snapshot": snapshot_record["snapshot_id"],
            "method": result["method"],
            "status": result["status"],
            "requested": metrics["requested_units"],
            "allocated": metrics["allocated_units"],
            "late": metrics["late_units"],
            "unfilled": metrics["unfilled_units"],
            "weighted_cost": result["evaluation"]["costs"]["total"],
            "gap_to_lp_bound": result["gap_to_lp_bound"],
            "elapsed_seconds": result.get("elapsed_seconds"),
        })
print("Source fingerprint:", report["source_fingerprint"])


In [ ]:
columns = ("snapshot", "method", "status", "requested", "allocated", "late", "unfilled", "weighted_cost")
table = "| " + " | ".join(columns) + " |\n"
table += "| " + " | ".join("---" for _ in columns) + " |\n"
for row in rows:
    table += "| " + " | ".join(str(row[column]) for column in columns) + " |\n"
display(Markdown(table))
print("Each snapshot is a separate current-state comparison. Repeated backlog is not summed as new demand.")


In [ ]:
opening = {row["method"]: row for row in rows if row["snapshot"] == snapshots[0].snapshot_id}
assert opening["milp"]["status"] == "optimal"
assert opening["milp"]["weighted_cost"] < opening["priority_cost_greedy"]["weighted_cost"]
for row in rows:
    assert row["allocated"] + row["unfilled"] == row["requested"]
    if row["method"] == "milp" and row["status"] == "optimal":
        assert abs(row["gap_to_lp_bound"]) < 1e-6
assert any(row["late"] > 0 for row in rows if row["snapshot"] == snapshots[1].snapshot_id)
print("Demand balance, LP/MILP optimality agreement, opportunity cost, and overdue accounting verified.")


## 4. Export a reviewable report and replace the fixture

The JSON retains provenance, cutoffs, assumptions, and all verified comparison results. The CSV exposes snapshot-level service and weighted costs without treating repeated snapshots as cumulative fulfillment. These plans remain advisory; historical dispatches are replayed input facts, not actions initiated by the notebook.

For a real pilot, obtain documented requested open backlog and physical movements in one consistent integer unit. Supply a complete SKU inventory observation at every cutoff, use actual observation timestamps, and reconcile all observations against the opening ledger. Run validation before solver comparison. Use separate business groups for policy tuning and holdout evaluation; this notebook neither calibrates weights nor establishes causal savings.


In [ ]:
output_dir = Path("business_pilot_results")
output_dir.mkdir(exist_ok=True)
(output_dir / "comparison.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
write_business_csv(report, output_dir / "comparison.csv")
print("Saved:", output_dir.resolve() / "comparison.json")
print("Saved:", output_dir.resolve() / "comparison.csv")
